In [ ]:
import os
import numpy as np
import pandas as pd


os.chdir(r'D:\Work\景观格局指标对PM2.5影响')

In [ ]:
targets = pd.read_excel("建模数据/因变量.xlsx", index_col=0, usecols=[0, 1])
targets

In [ ]:
features = pd.read_excel("建模数据/自变量/01-Spring.xlsx", index_col=0)
features

In [ ]:
features_scaled = features.apply(lambda x: (x - x.min()) / (x.max() - x.min()))

In [ ]:
def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

In [ ]:
def calcMrer(y_true, y_pred):
    """
    计算MRER
    :param y_true: 实际值数组
    :param y_pred: 预测值数组
    :return: MRER值
    """
    relative_errors = np.abs((y_pred - y_true) / y_true)
    return np.mean(relative_errors)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
        features_scaled[['ELP1200', 'SP', 'ILP1200']].values, targets.values.reshape(-1), train_size=0.8, test_size=0.2, random_state=1) 

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()

# 训练模型
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

In [ ]:
n = X_test.shape[0]
p =  X_test.shape[1]-1
print('系数:', model.coef_)
print('截距:', model.intercept_)
mse = mean_squared_error(y_test, y_pred)
print('标准估计误差see:', np.sqrt(mse * n / (n - p - 1)))
print('均方误差(MSE):', mse)
print('均方根误差(RMSE):', np.sqrt(mse))
r2 = r2_score(y_test, y_pred)
print("R2分数:", r2)
print('调整R2分数:', adjusted_r2(r2, n, p))
mrer = calcMrer(y_test, y_pred)
print("MRER: ", mrer)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
        features[['ELP1200', 'SP', 'ILP1200']].values, targets.values.reshape(-1), train_size=0.8, test_size=0.2, random_state=1) 

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

# 定义参数网格
param_grid = {
    # 'n_estimators': list(range(10, 200, 5)),
    # 'max_depth': list(range(1, 50, 2))
    'max_features': list(range(1, 50, 2))
}

# 创建网格搜索对象
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(
        n_estimators=75,
        max_depth=5,
        min_samples_leaf=2,
        min_samples_split=2,
        random_state=1),
    param_grid=param_grid,
    cv=5,
    n_jobs=24,
    verbose=2
)

# 执行网格搜索
grid_search.fit(X_train, y_train)

# 输出最佳参数
print("最佳参数:", grid_search.best_params_)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_regressor = RandomForestRegressor(n_estimators=75,
                                     max_depth=5,
                                     min_samples_leaf=2,
                                     min_samples_split=2,
                                     max_features=3,
                                     random_state=1)

# 训练模型
rf_regressor.fit(X_train, y_train)

# 预测
y_pred = rf_regressor.predict(X_test)

In [ ]:
n = X_test.shape[0]
p =  X_test.shape[1]-1
mse = mean_squared_error(y_test, y_pred)
print('标准估计误差see:', np.sqrt(mse * n / (n - p - 1)))
print('均方误差(MSE):', mse)
print('均方根误差(RMSE):', np.sqrt(mse))
r2 = r2_score(y_test, y_pred)
print("R2分数:", r2)
print('调整R2分数:', adjusted_r2(r2, n, p))
mrer = calcMrer(y_test, y_pred)
print("MRER: ", mrer)